# 第97章 K-Means客户分群实战

将 K-Means 用于客户分群，结合轮廓系数、簇规模和业务画像解释分群。


## 先解决一个小问题

围绕“K-Means客户分群实战”完成一个可验证的小型建模实验：先明确输入和目标，再比较方法带来的变化。将 K-Means 用于客户分群，结合轮廓系数、簇规模和业务画像解释分群。


## 这章为什么先学

这是“机器学习”建模主线中的第 97 章，重点放在“K-Means客户分群实战”对应的一个具体决策，而不是重复完整流程。


## 开始前确认

- 能够使用 pandas 读取、筛选和汇总数据
- 理解训练集、测试集和基本统计指标
- 本章会进一步练习：构造客户级 RFM 特征、标准化后聚类、比较多个簇数


## 做完要留下什么

完成一份围绕“K-Means客户分群实战”的可运行实验：包含数据准备、方法执行、指标或图表证据，以及一句有边界的结论。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 构造客户级 RFM 特征
- 标准化后聚类
- 比较多个簇数
- 输出可行动的簇画像


## 核心概念

- 目标：\(\min_\mu\sum_i\min_k||x_i-\mu_k||^2\)
- 簇编号没有大小含义
- 轮廓系数兼顾簇内紧密和簇间分离
- 分群稳定性比单次最优分数更重要


## 示例 1：数据与问题定义

先明确样本、特征、目标和验证方式，再训练模型。


In [ ]:
import pandas as pd

sales = pd.read_csv("/datasets/uci_online_retail_200k.csv", parse_dates=['InvoiceDate'])
sales = sales[(sales.Quantity>0)&(sales.UnitPrice>0)&sales.CustomerID.notna()].copy()
sales['revenue']=sales.Quantity*sales.UnitPrice; snapshot=sales.InvoiceDate.max()+pd.Timedelta(days=1)
rfm = sales.groupby('CustomerID').agg(recency=('InvoiceDate', lambda x:(snapshot-x.max()).days), frequency=('InvoiceNo', 'nunique'), monetary=('revenue', 'sum')).clip(lower=0)


## 示例 2：模型、公式与诊断

把核心数学量映射到 sklearn 输出，并检查泛化表现。


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
import numpy as np

features = np.log1p(rfm); Xs=StandardScaler().fit_transform(features)
scores = {}; models={}
for k in range(2,7):
    models[k]=KMeans(n_clusters=k, n_init=20, random_state=97).fit(Xs); scores[k]=silhouette_score(Xs, models[k].labels_)
best_k = max(scores, key=scores.get); rfm['cluster']=models[best_k].labels_
print('scores:', {k:round(v,3) for k, v in scores.items()})
display(rfm.groupby('cluster').agg(customers=('monetary', 'size'), recency=('recency', 'median'), frequency=('frequency', 'median'), monetary=('monetary', 'median')).round(1))


## 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 常见误区

- 在交易行而非客户粒度聚类
- 金额偏态不处理
- 把簇编号写成价值等级
- 没有验证不同随机种子下的稳定性


## 综合练习

1. 修改一个关键参数并重新运行
2. 记录指标变化并解释原因
3. 检查结论是否依赖测试集或隐藏泄漏

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“修改一个关键参数并重新运行”。
2. **独立完成**：不复制示例代码，完成“记录指标变化并解释原因”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“检查结论是否依赖测试集或隐藏泄漏”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
practice_sizes = rfm.cluster.value_counts()
print(practice_sizes.sort_index())

# 自检
assert practice_sizes.sum()==len(rfm)
assert practice_sizes.size==best_k


## 本章小结

将 K-Means 用于客户分群，结合轮廓系数、簇规模和业务画像解释分群。


### 你已经掌握

- 构造客户级 RFM 特征
- 标准化后聚类
- 比较多个簇数
- 输出可行动的簇画像


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 数据与问题定义 | 先明确样本、特征、目标和验证方式，再训练模型。 | `pd.read_csv()`、`sales.CustomerID.notna()`、`sales.InvoiceDate.max()`、`pd.Timedelta()` |
| 模型、公式与诊断 | 把核心数学量映射到 sklearn 输出，并检查泛化表现。 | `np.log1p()`、`scores.items()`、`rfm.groupby()`、`.fit_transform()` |


### 需要注意

- 在交易行而非客户粒度聚类
- 金额偏态不处理
- 把簇编号写成价值等级
- 没有验证不同随机种子下的稳定性


### 完成检查

- [ ] 能够构造客户级 RFM 特征
- [ ] 能够标准化后聚类
- [ ] 能够比较多个簇数
- [ ] 能够输出可行动的簇画像


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
